# US 10-day basket — Refinitiv trader notebook v2

This notebook does four jobs:

1. loads the current portfolio snapshot,
2. adds a small set of new US names,
3. pulls sector/country/history with Refinitiv,
4. builds a **simple 10-day basket** with strict risk constraints and a clear trade plan.

The philosophy is simple:

- cut the weak cyclicals and Europe-linked noise,
- keep only the US names that still deserve risk,
- add a small number of fresh US names that fit the tape,
- use hard constraints so the whole book cannot bleed at once again.


## Desk view

For the next 10 trading days, the clean expression is:

- **Power / utilities** for ballast and domestic demand,
- **one oil hedge**,
- **quality / stable demand**,
- **very small cyclical sleeve only if momentum confirms**.

That means:
- underweight or exit materials, transport, weak discretionary, Europe-linked names,
- overweight utilities, one energy hedge, and a small quality/staples sleeve,
- no averaging down,
- no large single-name bets.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import date, timedelta

plt.rcParams["figure.figsize"] = (11, 6)

CSV_PATH = Path("Open Positions Mar 8 2026.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("/mnt/data/Open Positions Mar 8 2026.csv")

df = pd.read_csv(CSV_PATH)

for col in ["Quantity", "LastPrice", "PricePaid", "DayChange", "MarketValue", "ProfitLossPercentage"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["ProfitLoss"] = (
    df["ProfitLoss"].astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip()
)
df["ProfitLoss"] = pd.to_numeric(df["ProfitLoss"], errors="coerce")

df["CostBasis"] = df["MarketValue"] - df["ProfitLoss"]
df["CurrentWeightPct"] = df["MarketValue"] / df["MarketValue"].sum() * 100

display(df[["Symbol","Description","MarketValue","CurrentWeightPct","ProfitLoss","ProfitLossPercentage"]].sort_values("ProfitLoss"))


## Refinitiv session

Use your own local Workspace / Refinitiv session.

If you use the newer package, change `refinitiv.data` to `lseg.data` and adapt the calls.


In [ ]:
# import refinitiv.data as rd
# rd.open_session()


## Candidate universe

Start from the current book, then add a few fresh US names that fit the current tape better.

Added candidates:
- `NEE` — utility / power demand
- `EXC` — utility / grid spend
- `XOM` — direct oil hedge
- `COST` — quality defensive consumer

You can add more later, but keep it small first.


In [ ]:
CURRENT_RIC_MAP = {
    "AD": "A.N",
    "EIX": "EIX.N",
    "FDX": "FDX.N",
    "FTI": "FTI.N",
    "GRMN": "GRMN.N",
    "HSY": "HSY.N",
    "HWM": "HWM.N",
    "MT": "MT.N",
    "OMC": "OMC.N",
    "PCG": "PCG.N",
    "SCCO": "SCCO.N",
    "TPL": "TPL.N",
    "TPR": "TPR.N",
}

NEW_RIC_MAP = {
    "NEE": "NEE.N",
    "EXC": "EXC.O",
    "XOM": "XOM.N",
    "COST": "COST.O",
}

RIC_MAP = {**CURRENT_RIC_MAP, **NEW_RIC_MAP}
BENCHMARK_RIC = "SPY.P"
LOOKBACK_DAYS = 120


## Robust history helper

This fixes the earlier `list index out of range` issue by not assuming a specific returned column name.


In [ ]:
def fetch_close_series(ric, start_date, end_date, debug=False):
    hist = rd.get_history(
        universe=ric,
        interval="daily",
        start=start_date,
        end=end_date,
    )

    if hist is None or hist.empty:
        raise ValueError(f"No history returned for {ric}")

    if debug:
        print(f"\n=== {ric} raw output ===")
        print(type(hist))
        print("index:", getattr(hist, "index", None))
        if hasattr(hist, "columns"):
            print("columns:", list(hist.columns))
        print(hist.head())

    if isinstance(hist, pd.Series):
        s = hist.copy()
        s.name = ric
        s.index = pd.to_datetime(s.index)
        return s.sort_index()

    hist = hist.copy().reset_index()
    hist.columns = [str(c) for c in hist.columns]

    date_candidates = [c for c in hist.columns if "date" in c.lower()]
    date_col = date_candidates[0] if date_candidates else hist.columns[0]

    preferred_patterns = ["close", "trdprc_1", "price close", "priceclose", "trade price"]
    price_col = None
    for c in hist.columns:
        cl = c.lower()
        if c != date_col and any(p in cl for p in preferred_patterns):
            price_col = c
            break

    if price_col is None:
        numeric_cols = hist.select_dtypes(include=[np.number]).columns.tolist()
        numeric_cols = [c for c in numeric_cols if c != date_col]
        if not numeric_cols:
            raise ValueError(f"{ric}: cannot identify price column. Columns={hist.columns.tolist()}")
        price_col = numeric_cols[-1]

    out = hist[[date_col, price_col]].copy()
    out.columns = ["Date", ric]
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out = out.dropna(subset=["Date", ric]).sort_values("Date")
    return out.set_index("Date")[ric]


In [ ]:
start_date = (date.today() - timedelta(days=LOOKBACK_DAYS)).isoformat()
end_date = date.today().isoformat()

series_list = []

for sym, ric in {**RIC_MAP, "_BENCH_": BENCHMARK_RIC}.items():
    try:
        s = fetch_close_series(ric, start_date, end_date, debug=False)
        series_list.append(s.rename(ric))
        print(f"OK: {ric} -> {len(s)} rows")
    except Exception as e:
        print(f"Failed: {ric} -> {e}")

if not series_list:
    raise ValueError("No series fetched. Check session / entitlements / RICs.")

close_wide = pd.concat(series_list, axis=1).sort_index().ffill().dropna(how="all")
display(close_wide.tail())


## Refinitiv metadata pull

This gives the live sector / industry / country tags.

If you get partial misses, keep the successfully resolved names and continue.


In [ ]:
meta = rd.get_data(
    universe=list(RIC_MAP.values()),
    fields=[
        "TR.CommonName",
        "TR.HeadquartersCountry",
        "TR.TRBCEconomicSector",
        "TR.TRBCBusinessSector",
        "TR.TRBCIndustryGroup",
        "TR.TRBCIndustry",
    ],
)

display(meta)


In [ ]:
meta = meta.copy()
meta.columns = [str(c) for c in meta.columns]

ric_col = meta.columns[0]
meta = meta.rename(columns={ric_col: "RIC"})

ric_to_symbol = {v: k for k, v in RIC_MAP.items()}
meta["Symbol"] = meta["RIC"].map(ric_to_symbol)

display(meta)


## Signal engine

Keep it simple:
- `Mom5`
- `Mom20`
- `Vol20`
- `DD20`
- `Beta20`

Then convert them into a small composite score.

For a 10-day book, I care most about:
- positive short-term momentum,
- manageable volatility,
- not being in an active drawdown.


In [ ]:
returns = close_wide.pct_change()

def max_drawdown(x):
    roll_max = x.cummax()
    dd = x / roll_max - 1
    return dd.min()

stat_rows = []
bench_ret = returns[BENCHMARK_RIC].dropna()

for sym, ric in RIC_MAP.items():
    if ric not in close_wide.columns:
        continue

    px = close_wide[ric].dropna()
    ret = px.pct_change().dropna()

    mom_5 = px.iloc[-1] / px.iloc[-6] - 1 if len(px) >= 6 else np.nan
    mom_20 = px.iloc[-1] / px.iloc[-21] - 1 if len(px) >= 21 else np.nan
    vol_20 = ret.tail(20).std() * np.sqrt(252) if len(ret) >= 20 else np.nan
    dd_20 = max_drawdown(px.tail(20)) if len(px) >= 20 else np.nan

    beta_20 = np.nan
    aligned = pd.concat([ret.rename("stock"), bench_ret.rename("bench")], axis=1).dropna()
    if len(aligned) >= 20:
        cov = np.cov(aligned["stock"].tail(20), aligned["bench"].tail(20))[0, 1]
        var = np.var(aligned["bench"].tail(20))
        beta_20 = cov / var if var != 0 else np.nan

    stat_rows.append({
        "Symbol": sym,
        "RIC": ric,
        "Mom5": mom_5,
        "Mom20": mom_20,
        "Vol20": vol_20,
        "DD20": dd_20,
        "Beta20": beta_20,
    })

signal_df = pd.DataFrame(stat_rows)

# simple score: momentum up, drawdown down, vol down
signal_df["Score"] = (
    1.4 * signal_df["Mom20"].fillna(-999)
    + 0.8 * signal_df["Mom5"].fillna(-999)
    - 0.35 * signal_df["Vol20"].fillna(signal_df["Vol20"].median())
    - 0.35 * signal_df["DD20"].abs().fillna(signal_df["DD20"].abs().median())
)

display(signal_df.sort_values("Score", ascending=False))


## Merge portfolio + metadata + signals


In [ ]:
full = pd.DataFrame({"Symbol": list(RIC_MAP.keys())})
full = full.merge(df[["Symbol","Description","Quantity","LastPrice","PricePaid","ProfitLoss","MarketValue","ProfitLossPercentage","CurrentWeightPct"]], on="Symbol", how="left")
full = full.merge(signal_df, on="Symbol", how="left")
full = full.merge(meta[["Symbol","TRBC Economic Sector","TRBC Business Sector","TRBC Industry Group","TRBC Industry","Headquarters Country"]], on="Symbol", how="left")

full["MarketValue"] = full["MarketValue"].fillna(0)
full["CurrentWeightPct"] = full["CurrentWeightPct"].fillna(0)

display(full.sort_values("Score", ascending=False))


## Hard exclusions

For this 10-day book, I do **not** want:
- Europe-linked names,
- materials / metals,
- transport,
- weak discretionary laggards.

That means default exits:
- `MT`
- `FTI`
- `GRMN`
- `SCCO`
- `FDX`
- `TPR`


In [ ]:
EXCLUDE = {"MT", "FTI", "GRMN", "SCCO", "FDX", "TPR"}

# Manual sector fallback in case metadata labels vary
SECTOR_FALLBACK = {
    "AD": "Health Care",
    "EIX": "Utilities",
    "PCG": "Utilities",
    "NEE": "Utilities",
    "EXC": "Utilities",
    "TPL": "Energy",
    "XOM": "Energy",
    "HSY": "Consumer Staples",
    "COST": "Consumer Staples / Defensive Retail",
    "HWM": "Industrials",
    "OMC": "Communication Services",
    "FDX": "Industrials - Transport",
    "SCCO": "Materials",
    "MT": "Materials",
    "TPR": "Consumer Discretionary",
    "FTI": "Energy Equipment",
    "GRMN": "Technology / Consumer Hardware",
}

full["SectorSimple"] = full["Symbol"].map(SECTOR_FALLBACK)
full["Eligible"] = ~full["Symbol"].isin(EXCLUDE)

eligible = full[full["Eligible"]].copy()
display(eligible[["Symbol","SectorSimple","CurrentWeightPct","Score","Mom5","Mom20","Vol20","DD20"]].sort_values("Score", ascending=False))


## Basket design

### My preferred basket
Keep it to **8 names + 8% cash**.

Core:
- `EIX`
- `PCG`
- `NEE`
- `TPL`
- `XOM`
- `AD`
- `HSY`
- `COST`

Optional small tactical alternates:
- `HWM`
- `OMC`
- `EXC`

I do **not** want all of them at once.  
I want the cleanest 8-name book.


In [ ]:
PREFERRED_ORDER = ["EIX", "PCG", "NEE", "TPL", "XOM", "AD", "HSY", "COST", "HWM", "OMC", "EXC"]

preferred = eligible[eligible["Symbol"].isin(PREFERRED_ORDER)].copy()
preferred["Rank"] = preferred["Symbol"].map({s:i for i, s in enumerate(PREFERRED_ORDER)})
preferred = preferred.sort_values(["Rank","Score"], ascending=[True, False])

display(preferred[["Symbol","SectorSimple","CurrentWeightPct","Score","Mom5","Mom20","Vol20","DD20"]])


## Final target weights

Simple, clean, capped.

- Utilities: strong but capped
- Energy: present but capped
- Staples / quality defense: meaningful but not oversized
- No weak macro-beta materials / transport sleeve
- 8% cash so the book can breathe


In [ ]:
TARGET_WEIGHTS = {
    "EIX": 12,
    "PCG": 12,
    "NEE": 11,
    "TPL": 11,
    "XOM": 10,
    "AD": 10,
    "HSY": 9,
    "COST": 9,
    "HWM": 0,
    "OMC": 0,
    "EXC": 8,   # alternate if you prefer 9 names; otherwise set to 0
}

# choose one of two simple profiles
PROFILE = "8_names_plus_cash"  # or "9_names_plus_cash"

if PROFILE == "8_names_plus_cash":
    TARGET_WEIGHTS["EXC"] = 0
    CASH_PCT = 16
else:
    CASH_PCT = 8

assert sum(TARGET_WEIGHTS.values()) + CASH_PCT == 100

book = full.copy()
book["TargetWeightPct"] = book["Symbol"].map(TARGET_WEIGHTS).fillna(0)
book["PortfolioValue"] = df["MarketValue"].sum()
book["TargetValue"] = book["PortfolioValue"] * book["TargetWeightPct"] / 100
book["DeltaValue"] = book["TargetValue"] - book["MarketValue"]
book["SharesToTrade"] = np.where(book["LastPrice"].fillna(0) > 0, np.round(book["DeltaValue"] / book["LastPrice"]), np.nan)

display(book[[
    "Symbol","SectorSimple","CurrentWeightPct","TargetWeightPct",
    "ProfitLossPercentage","Mom5","Mom20","Score","DeltaValue","SharesToTrade"
]].sort_values("TargetWeightPct", ascending=False))


## Constraints

These are the non-negotiables.

### Position constraints
- Max single-name weight = **12%**
- Top 3 combined <= **35%**
- Top 5 combined <= **56%**
- New position minimum = **8%** only if it makes the basket
- Cash sleeve = **8% to 16%**

### Sector constraints
- Utilities <= **35%**
- Energy <= **22%**
- Staples / defensive consumer <= **18%**
- Quality health care / tools <= **12%**
- Materials = **0%**
- Transport = **0%**
- Consumer discretionary = **0%**

### Execution rules
- No averaging down
- New buys in **2 tranches**
- Hard stop from average new entry = **-6%**
- Take-profit rule: trim **1/3** at **+7%**
- After **+7%**, trail the rest by **4.5%**
- Time stop: after **10 trading days**, exit any name with return < **+1%** and negative 5-day momentum

### Book-level kill switch
- If total basket drawdown from rebalance hits **-3.5%**, cut the weakest 2 names by half
- If total basket drawdown hits **-5%**, reduce gross exposure by another **10%**


## Entry plan

### Day 1
Sell to zero:
- `MT`
- `FTI`
- `GRMN`
- `SCCO`
- `FDX`
- `TPR`

Build **60% of each target** immediately in:
- `EIX`
- `PCG`
- `NEE`
- `TPL`
- `XOM`
- `AD`
- `HSY`
- `COST`

### Day 3 to Day 4
Add the remaining **40% of target** only if:
- stock 5-day momentum > 0
- stock 20-day momentum > 0
- stock closes above its 10-day moving average

If a name fails confirmation, do not force it. Leave the cash idle.


## Why these sectors

### Utilities — `EIX`, `PCG`, `NEE`
This is the stability spine.  
Power demand has been getting structural support from data centers and electrification.

### Energy — `TPL`, `XOM`
This is the macro hedge.  
If oil remains elevated, this sleeve helps offset the pressure that higher fuel prices put on the broader market.

### Quality / lower macro-beta — `AD`
Cleaner exposure than trying to rescue a broken cyclical.

### Staples / defensive consumer — `HSY`, `COST`
Not a place to swing for home runs.  
A place to lower left-tail damage in a shaky tape.

### What I do not want
- `MT`, `SCCO`: too much materials / macro beta
- `FDX`: wrong transport exposure when oil is volatile
- `TPR`: weak discretionary rebound trades are not the right use of risk right now
- `FTI`, `GRMN`: not aligned with the simplified US basket


In [ ]:
# quick charts
plot_df = book[book["TargetWeightPct"] > 0][["Symbol","CurrentWeightPct","TargetWeightPct"]].copy()
x = np.arange(len(plot_df))
w = 0.4

plt.figure(figsize=(12,6))
plt.bar(x - w/2, plot_df["CurrentWeightPct"], width=w, label="Current")
plt.bar(x + w/2, plot_df["TargetWeightPct"], width=w, label="Target")
plt.xticks(x, plot_df["Symbol"])
plt.title("Current vs target weights")
plt.ylabel("Weight %")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,6))
book.groupby("SectorSimple")["TargetWeightPct"].sum().sort_values(ascending=False).plot(kind="bar")
plt.title("Target sector exposure")
plt.ylabel("Weight %")
plt.tight_layout()
plt.show()


## Bottom line

This is not a rebound basket.  
It is a **survive-first, then compound** basket.

The edge here is not complexity.  
The edge is:

- simpler book,
- cleaner sectors,
- one macro hedge,
- smaller left tail,
- hard exits,
- no averaging down.
